In [1]:
def estrai_id_e_verso(header_line):
    """
    Estrae l'identificativo univoco e l'orientamento della read dall'header FASTQ.

    Args:
        header_line (str): La prima riga del record FASTQ (deve iniziare con '@').

    Returns:
        tuple: Una tupla contenente:
            - id_base (str): L'ID della read senza il prefisso '@' e senza il suffisso di orientamento.
            - verso (str o None): '1' per Forward, '2' per Reverse, o None se non identificato.
    """
    # Isola la prima stringa prima di eventuali spazi e rimuove la '@' iniziale
    id_completo = header_line.split()[0][1:]
    
    if id_completo.endswith("/1"):
        return id_completo[:-2], "1"
    elif id_completo.endswith("/2"):
        return id_completo[:-2], "2"
    
    return id_completo, None


def valida_record_fastq(header, seq, plus, qual):
    """
    Verifica l'integrità strutturale e biologica di un record FASTQ.

    Args:
        header (str): Riga di intestazione (atteso inizio con '@').
        seq (str): Sequenza nucleotidica.
        plus (str): Riga separatrice (atteso inizio con '+').
        qual (str): Stringa dei punteggi di qualità Phred.

    Returns:
        bool: True se il record rispetta gli standard FASTQ, False altrimenti.
    """
    return (
        header.startswith("@") and
        plus.startswith("+") and
        len(seq) > 0 and
        len(seq) == len(qual) and
        all(base in "ACGTN" for base in seq.upper())
    )


def leggi_e_valida_coppie(file_r1, file_r2):
    """
    Legge, sincronizza e valida in parallelo due file FASTQ Paired-End.

    La funzione processa i file leggendo 4 righe alla volta (1 record) per 
    ottimizzare l'uso della memoria. Verifica la corrispondenza degli ID 
    e l'integrità strutturale di ogni record.

    Args:
        file_r1 (str): Percorso del file FASTQ contenente le read Forward.
        file_r2 (str): Percorso del file FASTQ contenente le read Reverse.

    Returns:
        list: Una lista di tuple. Ogni tupla contiene due tuple annidate 
              (una per R1, una per R2) con le 4 stringhe del record FASTQ.

    Raises:
        ValueError: Se i file non sono sincronizzati, troncati o contengono record non validi.
    """
    coppie_pulite = []
    contatore_read = 0

    with open(file_r1, "r") as f1, open(file_r2, "r") as f2:
        
        while True:
            rec1 = [f1.readline().strip() for _ in range(4)]
            rec2 = [f2.readline().strip() for _ in range(4)]
            
            # Controllo EOF (End Of File) pulito
            if rec1[0] == "" and rec2[0] == "":
                break 
                
            # Controllo asimmetria numero di righe
            if (rec1[0] == "" and rec2[0] != "") or (rec1[0] != "" and rec2[0] == ""):
                raise ValueError(
                    f"[ERRORE CRITICO] Desincronizzazione file rilevata. "
                    f"I file hanno un numero differente di record all'altezza della read {contatore_read + 1}."
                )

            # Controllo file troncato (mancano righe all'interno di un record)
            if any(riga == "" for riga in rec1) or any(riga == "" for riga in rec2):
                raise ValueError(
                    f"[ERRORE STRUTTURALE] File troncato. Il record n° {contatore_read + 1} "
                    f"non contiene le 4 righe standard previste dal formato FASTQ."
                )
            
            contatore_read += 1
            h1, seq1, p1, q1 = rec1
            h2, seq2, p2, q2 = rec2
            
            # Validazione sintassi record
            if not valida_record_fastq(h1, seq1, p1, q1):
                raise ValueError(f"[ERRORE SINTASSI] Record non valido nel file R1 alla posizione {contatore_read}.")
            if not valida_record_fastq(h2, seq2, p2, q2):
                raise ValueError(f"[ERRORE SINTASSI] Record non valido nel file R2 alla posizione {contatore_read}.")
                
            # Controllo accoppiamento ID
            id1, verso1 = estrai_id_e_verso(h1)
            id2, verso2 = estrai_id_e_verso(h2)
            
            if id1 != id2:
                raise ValueError(
                    f"[ERRORE ACCOPPIAMENTO] Read non corrispondenti alla posizione {contatore_read}.\n"
                    f" -> Trovato ID R1: {id1}\n"
                    f" -> Trovato ID R2: {id2}"
                )
                
            # Avviso non bloccante su possibile inversione file input
            if verso1 == "2" or verso2 == "1":
                print(
                    f"[WARNING] Possibile inversione dei file input alla read {contatore_read}. "
                    f"R1 sembra contenere read Reverse e R2 read Forward."
                )
            
            coppia_validata = ((h1, seq1, p1, q1), (h2, seq2, p2, q2))
            coppie_pulite.append(coppia_validata)

    return coppie_pulite


In [2]:
# ==========================================
# TEST LOCALE
# ==========================================
if __name__ == "__main__":
    print("=== AVVIO VALIDAZIONE MODULO INPUT ===")
    
    file_prova_1 = "test_R1.fastq.txt"
    file_prova_2 = "test_R2.fastq.txt"
    
    try:
        print(f" Lettura in corso dei file:\n    - {file_prova_1}\n    - {file_prova_2}")
        risultato = leggi_e_valida_coppie(file_prova_1, file_prova_2)
        
        print("\n VALIDAZIONE COMPLETATA CON SUCCESSO")
        print(f" Totale coppie processate e validate: {len(risultato)}")
        print("\n Anteprima prima coppia strutturata:")
        
        # Stampiamo il risultato
        r1_es, r2_es = risultato[0]
        print(f"    R1 -> ID: {r1_es[0]} | Len Seq: {len(r1_es[1])} | Len Qual: {len(r1_es[3])}")
        print(f"    R2 -> ID: {r2_es[0]} | Len Seq: {len(r2_es[1])} | Len Qual: {len(r2_es[3])}")
        
    except Exception as e:
        print(f"\n[-] ESECUZIONE INTERROTTA:\n{e}")
        

=== AVVIO VALIDAZIONE MODULO INPUT ===
 Lettura in corso dei file:
    - test_R1.fastq.txt
    - test_R2.fastq.txt

 VALIDAZIONE COMPLETATA CON SUCCESSO
 Totale coppie processate e validate: 1

 Anteprima prima coppia strutturata:
    R1 -> ID: @Read_Misteriosa/1 | Len Seq: 8 | Len Qual: 8
    R2 -> ID: @Read_Misteriosa/2 | Len Seq: 8 | Len Qual: 8
